In [1]:
import glob
import polars as pl

files = glob.glob(
    "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/*.parquet"
)

print("Parquet files found:", len(files))

Parquet files found: 265


i

In [2]:
 # Read the first Parquet file
df = pl.read_parquet(files[0])

print("Rows:", df.height)
print("Columns:")
print(df.columns)

Rows: 25000
Columns:
['abstract', 'additional_entities', 'date_created', 'date_modified', 'description', 'event', 'identifier', 'image', 'in_language', 'infoboxes', 'is_part_of', 'license', 'main_entity', 'name', 'references', 'sections', 'tables', 'url', 'version']


In [3]:
# Check the important article fields

print("Sample article:")
print(df.select(["name", "url"]).head(5))

print("\nData types:")
print(df.select([
    "name",
    "sections",
    "infoboxes",
    "image",
    "references",
    "version"
]).schema)

Sample article:
shape: (5, 2)
┌─────────────────────────────┬─────────────────────────────────┐
│ name                        ┆ url                             │
│ ---                         ┆ ---                             │
│ str                         ┆ str                             │
╞═════════════════════════════╪═════════════════════════════════╡
│ Helmingham Dell             ┆ https://en.wikipedia.org/wiki/… │
│ Klaipėda Free Economic Zone ┆ https://en.wikipedia.org/wiki/… │
│ Swedish for immigrants      ┆ https://en.wikipedia.org/wiki/… │
│ Tiquadra syntripta          ┆ https://en.wikipedia.org/wiki/… │
│ The Mighty Jingles          ┆ https://en.wikipedia.org/wiki/… │
└─────────────────────────────┴─────────────────────────────────┘

Data types:
Schema({'name': String, 'sections': String, 'infoboxes': String, 'image': Struct({'content_url': String, 'height': Int64, 'width': Int64}), 'references': List(Struct({'identifier': String, 'metadata': String, 'source': Struct({'lin

In [4]:
# Extract the five Challenge 3 factors

def count_items(value):
    if value is None:
        return 0
    if isinstance(value, str):
        return len(value.splitlines())
    if isinstance(value, list):
        return len(value)
    return 0


def count_images(value):
    return 0 if value is None else 1


def count_infobox_fields(value):
    if not value:
        return 0
    return len(value.splitlines())


results = df.select([
    "name",
    "url",
    pl.col("version")
      .struct.field("number_of_characters")
      .alias("article_length"),
    pl.col("sections")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("section_count"),
    pl.col("infoboxes")
      .map_elements(count_infobox_fields, return_dtype=pl.Int64)
      .alias("infobox_field_count"),
    pl.col("image")
      .map_elements(count_images, return_dtype=pl.Int64)
      .alias("image_count"),
    pl.col("references")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("reference_count")
])

print(results.head())
print("\nArticles analyzed:", results.height)

shape: (5, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ name         ┆ url         ┆ article_len ┆ section_cou ┆ infobox_fie ┆ image_count ┆ reference_c │
│ ---          ┆ ---         ┆ gth         ┆ nt          ┆ ld_count    ┆ ---         ┆ ount        │
│ str          ┆ str         ┆ ---         ┆ ---         ┆ ---         ┆ i64         ┆ ---         │
│              ┆             ┆ i64         ┆ i64         ┆ i64         ┆             ┆ i64         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Helmingham   ┆ https://en. ┆ 3213        ┆ 1           ┆ 1           ┆ 1           ┆ 0           │
│ Dell         ┆ wikipedia.o ┆             ┆             ┆             ┆             ┆             │
│              ┆ rg/wiki/…   ┆             ┆             ┆             ┆             ┆             │
│ Klaipėda     ┆ https://en. ┆ 8861        ┆ 1           ┆ 1           ┆ 1   

In [5]:
# Replace missing factor values with 0

results = results.with_columns([
    pl.col("article_length").fill_null(0),
    pl.col("section_count").fill_null(0),
    pl.col("infobox_field_count").fill_null(0),
    pl.col("image_count").fill_null(0),
    pl.col("reference_count").fill_null(0)
])

print(results.head())

shape: (5, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ name         ┆ url         ┆ article_len ┆ section_cou ┆ infobox_fie ┆ image_count ┆ reference_c │
│ ---          ┆ ---         ┆ gth         ┆ nt          ┆ ld_count    ┆ ---         ┆ ount        │
│ str          ┆ str         ┆ ---         ┆ ---         ┆ ---         ┆ i64         ┆ ---         │
│              ┆             ┆ i64         ┆ i64         ┆ i64         ┆             ┆ i64         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Helmingham   ┆ https://en. ┆ 3213        ┆ 1           ┆ 1           ┆ 1           ┆ 0           │
│ Dell         ┆ wikipedia.o ┆             ┆             ┆             ┆             ┆             │
│              ┆ rg/wiki/…   ┆             ┆             ┆             ┆             ┆             │
│ Klaipėda     ┆ https://en. ┆ 8861        ┆ 1           ┆ 1           ┆ 1   

In [6]:
# Calculate percentile-based content-richness score

factors = {
    "article_length": 0.40,
    "section_count": 0.20,
    "reference_count": 0.20,
    "infobox_field_count": 0.10,
    "image_count": 0.10
}

for column, weight in factors.items():
    results = results.with_columns(
        (
            pl.col(column).rank(method="average") / results.height * 100
        ).alias(f"{column}_percentile")
    )

results = results.with_columns(
    (
        pl.col("article_length_percentile") * 0.40 +
        pl.col("section_count_percentile") * 0.20 +
        pl.col("reference_count_percentile") * 0.20 +
        pl.col("infobox_field_count_percentile") * 0.10 +
        pl.col("image_count_percentile") * 0.10
    ).alias("content_richness_score")
)

print(
    results.select([
        "name",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count",
        "content_richness_score"
    ]).head(10)
)

shape: (10, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ name         ┆ article_len ┆ section_cou ┆ infobox_fie ┆ image_count ┆ reference_c ┆ content_ric │
│ ---          ┆ gth         ┆ nt          ┆ ld_count    ┆ ---         ┆ ount        ┆ hness_score │
│ str          ┆ ---         ┆ ---         ┆ ---         ┆ i64         ┆ ---         ┆ ---         │
│              ┆ i64         ┆ i64         ┆ i64         ┆             ┆ i64         ┆ f64         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Helmingham   ┆ 3213        ┆ 1           ┆ 1           ┆ 1           ┆ 0           ┆ 44.4246     │
│ Dell         ┆             ┆             ┆             ┆             ┆             ┆             │
│ Klaipėda     ┆ 8861        ┆ 1           ┆ 1           ┆ 1           ┆ 0           ┆ 60.1678     │
│ Free         ┆             ┆             ┆             ┆             ┆    

In [7]:
# Identify relatively limited-content articles
# Bottom 5% of the score distribution

threshold = results.select(
    pl.col("content_richness_score").quantile(0.05)
).item()

results = results.with_columns(
    (pl.col("content_richness_score") <= threshold)
    .alias("limited_content")
)

print(f"Content-richness threshold: {threshold:.2f}")
print(
    "Limited-content articles:",
    results.filter(pl.col("limited_content")).height
)

Content-richness threshold: 29.90
Limited-content articles: 1251


In [8]:
# Create the same 10,000-article sample used by the project

sample_size = min(10000, results.height)

results = results.sample(
    n=sample_size,
    seed=42
)

print("Articles in final sample:", results.height)

Articles in final sample: 10000


In [9]:
# Recalculate the limited-content threshold for the 10,000-article sample

threshold = results.select(
    pl.col("content_richness_score").quantile(0.05)
).item()

results = results.with_columns(
    (pl.col("content_richness_score") <= threshold)
    .alias("limited_content")
)

print(f"Content-richness threshold: {threshold:.2f}")
print(
    "Limited-content articles:",
    results.filter(pl.col("limited_content")).height
)

Content-richness threshold: 29.99
Limited-content articles: 501


In [10]:
# Rank articles from lowest to highest content-richness score

results = results.sort("content_richness_score")

results = results.with_row_index("rank", offset=1)

print(
    results.select([
        "rank",
        "name",
        "content_richness_score",
        "article_length",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count",
        "limited_content"
    ]).head(10)
)

shape: (10, 9)
┌──────┬────────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ rank ┆ name       ┆ content_ri ┆ article_le ┆ … ┆ infobox_fi ┆ image_cou ┆ reference ┆ limited_c │
│ ---  ┆ ---        ┆ chness_sco ┆ ngth       ┆   ┆ eld_count  ┆ nt        ┆ _count    ┆ ontent    │
│ u32  ┆ str        ┆ re         ┆ ---        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│      ┆            ┆ ---        ┆ i64        ┆   ┆ i64        ┆ i64       ┆ i64       ┆ bool      │
│      ┆            ┆ f64        ┆            ┆   ┆            ┆           ┆           ┆           │
╞══════╪════════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 1    ┆ Arjun Ray  ┆ 23.875     ┆ 99         ┆ … ┆ 0          ┆ 0         ┆ 0         ┆ true      │
│ 2    ┆ Polish     ┆ 23.8766    ┆ 117        ┆ … ┆ 0          ┆ 0         ┆ 0         ┆ true      │
│      ┆ Uplanders  ┆            ┆            ┆   ┆            ┆           ┆

In [11]:
# Export the WikiWeak results

output_path = "/kaggle/working/wikiweak_results.csv"

results.write_csv(output_path)

print("============================================")
print("       WIKIWEAK EXPORT COMPLETE")
print("============================================")
print(f"Articles analyzed: {results.height}")
print(f"Limited-content threshold: {threshold:.2f}")
print(
    "Limited-content articles:",
    results.filter(pl.col("limited_content")).height
)
print(f"\nCreated: {output_path}")

       WIKIWEAK EXPORT COMPLETE
Articles analyzed: 10000
Limited-content threshold: 29.99
Limited-content articles: 501

Created: /kaggle/working/wikiweak_results.csv


In [12]:
# Example: inspect one limited-content article

limited_article = (
    results
    .filter(pl.col("limited_content") == True)
    .sort("content_richness_score")
    .row(0, named=True)
)

print("Article:", limited_article["name"])
print("Word Count:", limited_article["article_length"])
print("Sections:", limited_article["section_count"])
print("Infobox Fields:", limited_article["infobox_field_count"])
print("Images:", limited_article["image_count"])
print("References:", limited_article["reference_count"])
print("Content-Richness Score:", round(limited_article["content_richness_score"], 2))

print("\nConclusion:")
print("This article has relatively limited measured content")
print("based on the combined five-factor score.")

Article: Arjun Ray
Word Count: 99
Sections: 1
Infobox Fields: 0
Images: 0
References: 0
Content-Richness Score: 23.88

Conclusion:
This article has relatively limited measured content
based on the combined five-factor score.
